# MLP Regression: Fitting $y = \sin(x)$ with a 2-Layer Neural Network

Train a small MLP to approximate a noisy sine wave using the autograd engine and nn modules built from scratch.

In [ ]:
import sys
from pathlib import Path

# Add project root so we can import core.*
project_root = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.append(project_root)

import numpy as np
import matplotlib.pyplot as plt

from core.autograd import Value
from core.nn import (
    DataLoader,
    Linear,
    MSELoss,
    ReLU,
    Sequential,
    SGD,
    kaiming_uniform_,
)

## 1. Data Generation

Generate 200 points: $x \sim U[-3, 3]$, $y = \sin(x) + \epsilon$, $\epsilon \sim \mathcal{N}(0, 0.1)$.

Split 80/20 into training and validation sets.

In [ ]:
np.random.seed(42)

x_all = np.random.uniform(-3, 3, (200, 1))
y_all = np.sin(x_all) + 0.1 * np.random.randn(200, 1)
x_train, y_train = x_all[:160], y_all[:160]
x_val, y_val = x_all[160:], y_all[160:]

print(f"Train: {x_train.shape}, Val: {x_val.shape}")

In [ ]:
# Quick look at the data
plt.scatter(x_train, y_train, s=8, alpha=0.6, label="train")
plt.scatter(x_val, y_val, s=8, alpha=0.6, label="val")
x_smooth = np.linspace(-3, 3, 300).reshape(-1, 1)
plt.plot(x_smooth, np.sin(x_smooth), "k--", label="true sin(x)")
plt.legend()
plt.title("Data");

## 2. DataLoader

Wrap training data in a `DataLoader` for mini-batch iteration.
Use `batch_size=16` with shuffling.

In [ ]:
train_loader = DataLoader(x_train, y_train, batch_size=16, shuffle=True)

print(f"Batches per epoch: {len(train_loader)}")

## 3. Model Definition

Architecture: `Linear(1, 16) -> ReLU() -> Linear(16, 1)`

Apply Kaiming Uniform initialization (gain=$\sqrt{2}$ for ReLU) to both Linear layers.

In [ ]:
model = Sequential(
    [
        Linear(1, 16),
        ReLU(),
        Linear(16, 1),
    ]
)
for p in model.parameters():
    if p.data.ndim == 2:
        kaiming_uniform_(p, gain=np.sqrt(2))

print(model)

## 4. Loss Function & Optimizer

In [ ]:
loss_fn = MSELoss()
optim = SGD(model.parameters(), lr=0.01)

## 5. Training Loop

For each epoch:
1. Iterate over mini-batches from the DataLoader
2. Forward: compute predictions and loss
3. Backward: `optim.zero_grad()` → `loss.backward()` → `optim.step()`
4. Compute validation loss on the full val set
5. Track both losses for plotting

In [ ]:
epochs = 1000
train_losses = []
val_losses = []

for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        bx, by = Value(batch_x), Value(batch_y)
        pred = model(bx)
        loss = loss_fn(pred, by)

        optim.zero_grad()
        loss.backward()
        optim.step()

        epoch_loss += loss.data

    train_losses.append(epoch_loss / len(train_loader))

    vx, vy = Value(x_val), Value(y_val)
    vpred = model(vx)
    vloss = loss_fn(vpred, vy)
    val_losses.append(vloss.data)

print(f"Final train loss: {train_losses[-1]:.6f}")
print(f"Final val loss:   {val_losses[-1]:.6f}")

## 6. Loss Curve

Plot training and validation loss over epochs.

In [ ]:
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.title("Training Progress")

## 7. Fitted Curve

Plot the learned function against the ground truth $\sin(x)$ and training data.

In [ ]:
x_dense = np.linspace(-3, 3, 300).reshape(-1, 1)
model.eval()  # not strictly needed here, but good practice
y_pred = model(Value(x_dense))

plt.scatter(x_train, y_train, s=8, alpha=0.4, label="train data")
plt.plot(x_dense, np.sin(x_dense), "k--", label="true sin(x)")
plt.plot(x_dense, y_pred.data, "r-", linewidth=2, label="MLP fit")
plt.legend()
plt.title("MLP Regression: sin(x)")
plt.ylim(-1.5, 1.5)